# CLOAF on Google Colab (GPU)

Use this when you have no university GPU yet. **Runtime → Change runtime type → GPU (T4 or better).**

Before you start, put the licensed data in your Google Drive (never in the repo):

```
MyDrive/cloaf_data/
  smpl/SMPL_NEUTRAL.pkl        (+ SMPL_MALE.pkl, SMPL_FEMALE.pkl)
  amass/CMU.tar.bz2            (or any other AMASS sub-dataset, "SMPL+H G" version)
```

Colab disconnects after a few hours: the checkpoint is saved to Drive, so you can resume from the last step.

In [ ]:
!nvidia-smi

In [ ]:
!git clone https://github.com/arhannlp/cloaf_impl.git
%cd cloaf_impl
!pip install -q torchdiffeq smplx trimesh

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/cloaf_data'
!mkdir -p data/body_models data/amass
!cp -r {DATA}/smpl data/body_models/
!for f in {DATA}/amass/*.tar.bz2; do tar -xjf "$f" -C data/amass; done
!ls data/body_models/smpl data/amass

## Step 0 – smoke test (fake body, should print SMOKE TEST OK)

In [ ]:
!python tests/smoke_test.py 2>&1 | grep -v -i warning

## Step 1 – Jacobian check on the real SMPL (compare with Fig. 5 / B.7 of the paper)

In [ ]:
!python scripts/check_linearity.py --smpl data/body_models/smpl --restarts 3

## Step 2 – simple field, no training (hand-target distance should shrink)

In [ ]:
!python scripts/demo_target_field.py --smpl data/body_models/smpl

## Step 3 – prepare AMASS

In [ ]:
!python scripts/prepare_amass.py --amass_root data/amass --out data/amass_smpl_30fps.npz

## Step 4 – train the motion field (time budget in minutes)

Watch the loss: it must go down. The checkpoint is saved every 1000 iterations and at the end.

In [ ]:
!python -m cloaf.train --smpl data/body_models/smpl --amass data/amass_smpl_30fps.npz \
    --batch 16 --minutes 150 --out checkpoints/field.pt
!mkdir -p {DATA}/checkpoints && cp checkpoints/field.pt {DATA}/checkpoints/

## Step 5 – run CLOAF on one sequence with injected self-intersections

Pick any file from data/amass. Results are copied to Drive; watch them locally with
`python scripts/view_results.py --smpl data/body_models/smpl --result sequence.npz`.

In [ ]:
import glob
seq = sorted(glob.glob('data/amass/**/*_poses.npz', recursive=True))[0]
print(seq)
!python scripts/run_on_sequence.py --smpl data/body_models/smpl --seq "{seq}" \
    --field checkpoints/field.pt --frames 90 --corrupt --mode all --out results/sequence_all.npz
!python scripts/run_on_sequence.py --smpl data/body_models/smpl --seq "{seq}" \
    --field checkpoints/field.pt --frames 90 --corrupt --mode colliding_only --out results/sequence_colliding_only.npz
!mkdir -p {DATA}/results && cp results/*.npz {DATA}/results/